# 📘 반도체 패키징 역설계 핵심 기법 & 알고리즘 해설서

이 문서는 AI 기반 반도체 패키징 역설계 파이프라인(Step 1 ~ Step 5)에 사용된 데이터 사이언스 및 전산역학 기법들을 입문자의 눈높이에 맞추어 비유와 함께 정리한 해설서입니다.

---

## [Step 1] 대리 모델 학습 및 데이터 증강

### 1. 절댓값 Max Peak 추출
* **어떤 기법인가요?** 시간에 따라 굽이치는 파형 데이터에서 가장 충격이 큰 '최고점'을 하나만 집어내는 전처리 기법입니다.
* **어떤 특징이 있나요?** 단순히 수치가 가장 큰 값(max)을 찾는 것이 아니라, 부호(+, -)를 떼고 파괴력(절댓값)이 제일 큰 곳을 찾은 뒤 **원래의 부호를 살려서** 가져옵니다.
* **작동 원리:** `[10, -30, 20]`이라는 파동이 있다면, 절댓값이 가장 큰 30의 위치를 찾고 원본값인 `-30`을 최종 결과로 선택합니다.
* **사용 이유:** 패키징이 냉각될 때 기판이 아래로 휘며 발생하는 치명적인 압축 응력(-)을 단순 최댓값(max) 연산으로 놓치는 오류를 막고 물리적 진실을 보존하기 위해 사용했습니다.

### 2. 피어슨 상관계수 (Pearson Correlation)
* **어떤 기법인가요?** 두 데이터가 얼마나 찰떡같이 서로를 따라다니는지(선형 관계)를 -1에서 1 사이의 숫자로 보여주는 데이터 분석의 기초 기법입니다.
* **어떤 특징이 있나요?** 1에 가까울수록 같이 커지고, -1에 가까울수록 반대로 움직이며, 0이면 서로 아무 관계가 없습니다.
* **작동 원리:** 두 변수가 평균으로부터 떨어져 있는 정도(분산)를 곱하여 수치화합니다.
* **사용 이유:** AI 학습 전에 "덮개(P5)를 두껍게 하면 휨(WarpMax)이 커지는구나"와 같은 공학적 인과관계를 엔지니어가 눈으로 직접 확인하고 검증하기 위해 사용했습니다.

### 3. GPR (가우시안 프로세스 회귀)
* **어떤 기법인가요?** 데이터를 점으로 찍어놓고, 그 점들을 지날 수 있는 **'무수히 많은 부드러운 곡선들의 확률'**을 통째로 예측하는 고급 AI 알고리즘입니다.
* **어떤 특징이 있나요?** 정답만 딱 뱉어내는 것이 아니라, **"내가 이 예측을 얼마나 확신하는가?"를 나타내는 오차 범위(불확실성, $\mu \pm \sigma$)**를 같이 알려줍니다.
* **사용 이유:** 트리 모델(XGBoost 등)은 모르는 영역(외삽)의 데이터를 만나면 양 끝단에 뾰족한 벽(Clipping 뿔)을 만드는 구조적 한계가 있습니다. 이를 극복하고 부드럽고 연속적인 물리적 응력 곡면을 모사하기 위해 도입했습니다.

### 4. ARD 커널 (Automatic Relevance Determination)
* **어떤 기법인가요?** GPR 모델이 스스로 **"어떤 설계 변수가 중요하고 어떤 변수가 쓸모없는지" 깨닫게 만드는 똑똑한 돋보기(커널)**입니다.
* **작동 원리:** 결과에 영향이 없는 변수 쪽으로는 눈금을 쭉 늘려버려 값의 변화를 무시하게 만들고, 예민한 변수는 눈금을 좁혀 미세한 변화도 현미경처럼 캐치하게 만듭니다.
* **사용 이유:** 6개의 두께(P1~P6) 중 휨과 박리를 지배하는 진짜 범인(예: P5)을 AI가 수학적으로 찾아내어 인간에게 증명하도록 만들기 위해 사용했습니다.

### 5. 라틴 하이퍼큐브 샘플링 (LHS)
* **어떤 기법인가요?** 다차원 공간에 점들을 **가장 겹치지 않고 골고루 흩뿌리는 고급 난수 생성 기법**입니다.
* **작동 원리:** 스도쿠 퍼즐처럼 각 차원을 바둑판으로 쪼갠 뒤, 어떤 가로/세로 줄에도 겹침 없이 점이 1개씩만 들어가도록 자리를 배치합니다.
* **사용 이유:** AI에게 물어볼 10만 개의 가상 조합을 만들 때, 특정 두께만 편식해서 생성되지 않고 6차원 탐색 공간 전체를 촘촘하게 찌르기 위해서입니다.

---

## [Step 2] 은닉 제약조건 분류기 (Gatekeeper)

### 6. 랜덤 포레스트 (Random Forest)
* **어떤 기법인가요?** 수백 개의 얕은 '스무고개 질문(결정 트리)'을 숲(Forest)처럼 모아서 다수결로 결론을 내리는 분류 알고리즘입니다.
* **어떤 특징이 있나요?** 하나의 뛰어난 천재보다 수백 명의 평범한 사람들이 모였을 때 더 정확하다는 '집단 지성(앙상블)'을 이용합니다.
* **사용 이유:** 시뮬레이션 격자(Mesh)가 깨지며 발산하는 물리적 파괴 조합의 경계선은 매우 불규칙합니다. 이 복잡한 지뢰밭의 패턴을 가장 잘 찾아내는 알고리즘이 랜덤 포레스트입니다.

### 7. 클래스 불균형 해소 (Class Weight Balancing)
* **어떤 기법인가요?** 데이터의 비율이 무너져 있을 때(예: 정상 95%, 불량 5%), AI가 "전부 정상이라고 찍으면 95점이네?"라고 게으름 피우는 것을 막는 기법입니다.
* **작동 원리:** 불량(Fail) 데이터를 틀렸을 때 정상(Safe)을 틀렸을 때보다 훨씬 더 가혹한 벌점(Weight)을 주어 학습 밸런스를 강제로 맞춥니다.
* **사용 이유:** 전체 데이터 중 해석이 터진 위험한 데이터가 훨씬 적기 때문에, 이 파탄 조합들을 하나라도 놓치지 않고 깐깐하게 걸러내기 위해서입니다.

---

## [Step 3] 파레토 프론티어 타겟 곡선 추출

### 8. 파레토 비지배 정렬 (Pareto Non-dominated Sorting)
* **어떤 기법인가요?** "싸면서도 성능 좋은 컴퓨터"처럼 상충하는 두 마리 토끼를 다 잡아야 할 때, **누구에게도 꿀리지 않는 '절대적 1등 그룹'을 솎아내는 수학적 필터**입니다.
* **작동 원리:** A라는 설계가 B보다 휨도 적고 박리도 적다면 "A가 B를 지배(Dominate)한다"고 판정합니다. 이렇게 1:1 대결을 벌여 아무에게도 지배당하지 않는 녀석들만 모아 'Frontier 0(1등 그룹)'으로 묶습니다.
* **사용 이유:** AI에게 "이 파동대로 역설계해 줘"라고 던져줄, 가성비가 가장 완벽한 물리적 '정답지(유토피아 타겟)'를 원본 데이터 속에서 발굴하기 위함입니다.

---

## [Step 4] 1D-CNN 오토인코더 잠재 매핑 역설계

### 9. 1D-CNN 오토인코더 (Autoencoder)
* **어떤 기법인가요?** 방대한 데이터를 핵심만 남겨 아주 작게 압축(Encoder)했다가, 다시 원래대로 복원(Decoder)하는 훈련을 통해 **'데이터의 엑기스 요약본'을 만드는 딥러닝 기술**입니다.
* **사용 이유:** 4,319개나 되는 복잡한 파동 숫자를 한 번에 6개의 두께(P)로 역추적하면 AI가 길을 잃습니다. 그래서 파동을 32개의 '핵심 뼈대(잠재 벡터)'로 압축한 뒤, 그 뼈대를 보고 두께를 유추하는 안정적인 2단계 매핑 작전을 쓰기 위해 도입했습니다.

### 10. 다층 퍼셉트론 (MLP) 역매핑
* **어떤 기법인가요?** 인간의 뇌 신경망을 모방하여, 입력과 출력 사이의 복잡한 규칙을 알아서 찾아내는 기초 딥러닝 모델입니다.
* **사용 이유:** 오토인코더가 뽑아준 32개의 압축 암호를 보고, "아! 이런 파동이 나오려면 기판(P1) 두께는 0.8, 덮개(P5)는 1.5여야 해"라고 치수로 번역해 주는 역설계 통역기로 사용했습니다.

---

## [Step 5] NSGA-II 기반 통합 강건 최적화

### 11. NSGA-II (다목적 유전 알고리즘)
* **어떤 기법인가요?** 다윈의 진화론(교배, 돌연변이, 적자생존)을 모방하여 수백 세대에 걸쳐 최고의 조합을 찾아내는 생물학적 최적화 알고리즘입니다.
* **사용 이유:** 딥러닝(Step 4)이 던져준 초안은 완벽하지 않으므로, 이 초안의 주변 공간(±10%)에서 치수를 조금씩 바꾸며(돌연변이) 현실적으로 흠결이 없는 완벽한 최종 치수를 다듬기 위함입니다.

### 12. 강건 최적화 (Robust Optimization, $\mu + 2\sigma < Limit$)
* **어떤 기법인가요?** AI가 틀릴 가능성(오차 범위)까지 모두 계산에 넣어, **최악의 상황에서도 파괴되지 않도록 보수적으로 설계**하는 엔지니어링 철학입니다.
* **작동 원리:** AI가 "응력이 100 나올 것 같아($\mu$)"라고 예측해도, "오차가 $\pm 10$ 이니까($\sigma$), 최악의 경우인 120($\mu + 2\sigma$)이 한계선을 넘는지" 가혹하게 검사합니다.
* **사용 이유:** AI의 예측을 100% 믿고 한계선에 아슬아슬하게 맞췄다가 실제 시뮬레이션에서 터지는 대참사를 원천 차단하기 위해서입니다.

### 13. Feasibility Rule (`pymoo` `G` Matrix)
* **어떤 기법인가요?** 유전 알고리즘이 규칙을 어겼을 때 단순히 실격 처리하는 대신, **"얼마나 심하게 어겼는지"를 정교하게 비교하여 올바른 진화의 방향을 알려주는 방식**입니다.
* **사용 이유:** 위반 시 묻지마식 대형 벌점(+999,999)을 주면 AI가 "어디로 가야 살 수 있는지" 방향을 잃습니다. 살짝 어긴 개체를 서서히 안전한 구역으로 유도하여 부드럽게 수렴시키기 위해 도입했습니다.

### 14. Knee Point (최적 밸런스 점 추출)
* **어떤 기법인가요?** 파레토 1등 곡선(활 모양) 중에서도 가장 볼록하게 튀어나온 '무릎(Knee)' 부분을 콕 집어내는 최종 선택 기법입니다.
* **사용 이유:** 휨만 극단적으로 줄이거나 박리만 줄인 편향된 설계가 아니라, 두 가지 치명적 불량 요소를 가장 가성비 좋게 방어하는 **'최고의 황금 밸런스 조합 1개'**를 최종 추천하기 위해서입니다.